---

In [1]:
# use_windowing_cropping_CORRECTED.py
import os
import nibabel as nib
import numpy as np
import imageio
import scipy.ndimage as ndimage
import glob

def resample(image, spacing, new_spacing=[1, 1], mode='constant'):
    new_shape = np.round(image.shape * (np.array(spacing) / np.array(new_spacing)))
    resize_factor = new_shape / image.shape
    image = ndimage.zoom(image, resize_factor, mode=mode)
    return image, resize_factor

def get_central_coordinates(scan, crop_dim):
    x, y = np.nonzero(scan)
    
    if len(x) == 0 or len(y) == 0:
        xm = scan.shape[0] // 2
        hm = scan.shape[1] // 2
    else:
        xl, xr = x.min(), x.max()
        yl, yr = y.min(), y.max()
        xm = int((xr + xl) / 2)
        hm = int((yr + yl) / 2)
    
    if xm < crop_dim:
        xm = crop_dim
    if hm < crop_dim:
        hm = crop_dim
    if xm > (scan.shape[0] - crop_dim):
        xm = scan.shape[0] - crop_dim
    if hm > (scan.shape[1] - crop_dim):
        hm = scan.shape[1] - crop_dim
    
    return (xm, hm)

def crop_scan(scan, xm, hm, crop_dim):
    return scan[xm-crop_dim:xm+crop_dim, hm-crop_dim:hm+crop_dim]

def process_scan(scan_path, mask_path, crop_dim=90, min_interval=-50, max_interval=300):
    
    scan = nib.load(scan_path)
    mask = nib.load(mask_path)
    
    scan_data = scan.get_fdata()
    mask_data = mask.get_fdata()
    
    # Binarize mask (convert 255 to 1)
    mask_data = (mask_data > 0).astype(float)
    
    # IMPROVED: Two-pass slice selection
    # Pass 1: Find slices with tumor
    mask_pixels_per_slice = []
    for i in range(scan_data.shape[2]):
        mask_pixels_per_slice.append(np.count_nonzero(mask_data[:, :, i]))
    
    # Get slices that have tumor
    slices_with_tumor = [i for i, count in enumerate(mask_pixels_per_slice) if count > 0]
    
    if not slices_with_tumor:
        print(f"  ⚠️ WARNING: No tumor found in mask!")
        return None, 0, 0
    
    # Pass 2: Among tumor slices, find one with most pixels in HU window
    pixels_by_slice = []
    for i in slices_with_tumor:
        slice_scan = scan_data[:, :, i]
        slice_mask = mask_data[:, :, i]
        
        # Apply mask
        masked_scan = np.where(slice_mask == 0, 0, slice_scan)
        
        # Apply HU window
        windowed = np.where(masked_scan < min_interval, 0, masked_scan)
        windowed = np.where(windowed > max_interval, 0, windowed)
        
        pixels_by_slice.append(np.count_nonzero(windowed))
    
    # Select best slice from tumor slices
    best_idx = np.argmax(pixels_by_slice)
    ct_slice_idx = slices_with_tumor[best_idx]
    tumor_pixel_count = pixels_by_slice[best_idx]
    
    if tumor_pixel_count == 0:
        print(f"  ⚠️ WARNING: No tumor pixels in HU window!")
        print(f"     Tumor exists on slices: {slices_with_tumor[0]}-{slices_with_tumor[-1]}")
        return None, 0, 0
    
    # Get the chosen slice
    ct_slice = scan_data[:, :, ct_slice_idx]
    mask_slice = mask_data[:, :, ct_slice_idx]
    
    # Apply Gaussian filter
    ct_slice = ndimage.gaussian_filter(ct_slice, sigma=0.5, order=0, truncate=3)
    ct_slice = np.where(mask_slice == 0, 0, ct_slice)
    
    # Get tumor center
    (xm, hm) = get_central_coordinates(ct_slice, crop_dim)
    
    # Resample if needed
    original_spacing = list(scan.header.get_zooms())[0:2]
    if original_spacing != [1.0, 1.0]:
        ct_slice, factor = resample(ct_slice, original_spacing, [1.0, 1.0], mode='nearest')
        mask_slice, _ = resample(mask_slice, original_spacing, [1.0, 1.0])
        mask_slice = np.where(mask_slice > 0.1, 1, 0)
    else:
        factor = [1.0, 1.0]
    
    ct_slice = np.where(mask_slice == 0, 0, ct_slice)
    
    # Crop
    ct_slice = crop_scan(ct_slice, int(xm*factor[0]), int(hm*factor[1]), crop_dim)
    mask_slice = crop_scan(mask_slice, int(xm*factor[0]), int(hm*factor[1]), crop_dim)
    
    # HU windowing and normalization
    ct_slice = ((ct_slice - min_interval) / (max_interval - min_interval)) * 255
    ct_slice = np.clip(ct_slice, 0, 255)
    ct_slice = np.where(mask_slice == 0, 0, ct_slice)
    
    # Convert to 8-bit
    ct_slice = np.rint(ct_slice).astype(np.uint8)
    
    return ct_slice, ct_slice_idx, tumor_pixel_count

# === MAIN ===
print("\n" + "="*70)
print("🖼️  CMC DATASET PREPROCESSING - CORRECTED")
print("="*70 + "\n")

NIFTI_FOLDER = "./convert_nifti_format"
OUTPUT_PATH = "./cmc_final_png"
os.makedirs(OUTPUT_PATH, exist_ok=True)

patient_folders = sorted([d for d in os.listdir(NIFTI_FOLDER) 
                         if os.path.isdir(os.path.join(NIFTI_FOLDER, d))])

print(f"Processing {len(patient_folders)} patients\n")
print("-"*70)

successful = 0
failed = 0

for i, patient_id in enumerate(patient_folders, 1):
    patient_dir = os.path.join(NIFTI_FOLDER, patient_id)
    
    print(f"[{i:3d}/{len(patient_folders)}] Patient: {patient_id}")
    
    scan_path = os.path.join(patient_dir, "image_re.nii.gz")
    mask_files = glob.glob(os.path.join(patient_dir, "mask_*_re.nii.gz"))
    
    if not os.path.exists(scan_path) or not mask_files:
        print(f"           ⚠️  Missing files\n")
        failed += 1
        continue
    
    mask_path = mask_files[0]
    
    try:
        ct_slice, slice_idx, tumor_pixels = process_scan(scan_path, mask_path)
        
        if ct_slice is not None:
            output_file = f"{patient_id}.png"
            imageio.imwrite(os.path.join(OUTPUT_PATH, output_file), ct_slice)
            print(f"           📍 Slice: {slice_idx} | Tumor pixels: {tumor_pixels}")
            print(f"           ✅ Saved: {output_file}\n")
            successful += 1
        else:
            print(f"           ❌ Could not process\n")
            failed += 1
    except Exception as e:
        print(f"           ❌ Error: {e}\n")
        import traceback
        traceback.print_exc()
        failed += 1

print("="*70)
print("📊 FINAL SUMMARY")
print("="*70)
print(f"Total patients: {len(patient_folders)}")
print(f"✅ Successfully processed: {successful}")
print(f"❌ Failed: {failed}")
print(f"\n✅ Final PNG images (180×180) saved to:")
print(f"   {os.path.abspath(OUTPUT_PATH)}")
print(f"\n🎉 PREPROCESSING COMPLETE!")
print("="*70 + "\n")


🖼️  CMC DATASET PREPROCESSING - CORRECTED

Processing 102 patients

----------------------------------------------------------------------
[  1/102] Patient: 003262P
           📍 Slice: 141 | Tumor pixels: 600
           ✅ Saved: 003262P.png

[  2/102] Patient: 005121C
           📍 Slice: 70 | Tumor pixels: 211
           ✅ Saved: 005121C.png

[  3/102] Patient: 009644P
           📍 Slice: 163 | Tumor pixels: 120
           ✅ Saved: 009644P.png

[  4/102] Patient: 016874P
           📍 Slice: 75 | Tumor pixels: 451
           ✅ Saved: 016874P.png

[  5/102] Patient: 020234P
           📍 Slice: 60 | Tumor pixels: 886
           ✅ Saved: 020234P.png

[  6/102] Patient: 021617P
           📍 Slice: 85 | Tumor pixels: 528
           ✅ Saved: 021617P.png

[  7/102] Patient: 022834D
           📍 Slice: 63 | Tumor pixels: 418
           ✅ Saved: 022834D.png

[  8/102] Patient: 027064G
           📍 Slice: 80 | Tumor pixels: 114
           ✅ Saved: 027064G.png

[  9/102] Patient: 033839P
       

---

### **`Diagnostic Script - Find the Problem:`**

**For one patient it doesn't able to convert into png why i don't understand it say no tumour pixels found even though when i open that patient into 3D slicer i'm able to see the tumour.**

There could be several reasons:

 1. Mask values are different (not 0 and 255)
 
 2. HU window is too narrow (tumor pixels are outside -50 to 300 range)
 
 3. Mask/scan misalignment


In [11]:
# debug_patient.py
"""
Debug script to find why a patient has "no tumor pixels"
"""

import nibabel as nib
import numpy as np
import glob

# UPDATE THIS to the problematic patient ID
PATIENT_ID = "965445P"  # e.g., "003262P"

patient_dir = f"./convert_nifti_format/{PATIENT_ID}"

print(f"\n{'='*70}")
print(f"🔍 DEBUGGING PATIENT: {PATIENT_ID}")
print(f"{'='*70}\n")

# Load files
scan_path = f"{patient_dir}/image_re.nii.gz"
mask_files = glob.glob(f"{patient_dir}/mask_*_re.nii.gz")

if not mask_files:
    print("❌ No mask file found!")
    exit(1)

mask_path = mask_files[0]

print(f"📂 Files:")
print(f"   Scan: {scan_path}")
print(f"   Mask: {mask_path}\n")

scan = nib.load(scan_path)
mask = nib.load(mask_path)

scan_data = scan.get_fdata()
mask_data = mask.get_fdata()

print(f"📐 Shapes:")
print(f"   Scan: {scan_data.shape}")
print(f"   Mask: {mask_data.shape}")

if scan_data.shape != mask_data.shape:
    print("   ⚠️  WARNING: Shapes don't match!\n")
else:
    print("   ✅ Shapes match\n")

# Check mask values
print(f"🎭 Mask Analysis:")
print(f"   Unique values: {np.unique(mask_data)}")
print(f"   Min: {mask_data.min()}")
print(f"   Max: {mask_data.max()}")
print(f"   Total non-zero pixels: {np.count_nonzero(mask_data)}")

# Count non-zero pixels per slice
mask_pixels_per_slice = []
for i in range(mask_data.shape[2]):
    mask_pixels_per_slice.append(np.count_nonzero(mask_data[:, :, i]))

slices_with_tumor = [i for i, count in enumerate(mask_pixels_per_slice) if count > 0]
print(f"   Slices with tumor: {len(slices_with_tumor)}/{mask_data.shape[2]}")
print(f"   Slice range: {min(slices_with_tumor) if slices_with_tumor else 'N/A'} to {max(slices_with_tumor) if slices_with_tumor else 'N/A'}")

if slices_with_tumor:
    max_tumor_slice = mask_pixels_per_slice.index(max(mask_pixels_per_slice))
    print(f"   Largest tumor on slice: {max_tumor_slice} ({mask_pixels_per_slice[max_tumor_slice]} pixels)\n")
else:
    print("   ❌ NO TUMOR FOUND IN ANY SLICE!\n")
    exit(1)

# Check scan values on tumor slice
print(f"🔬 Scan Analysis (slice {max_tumor_slice}):")
scan_slice = scan_data[:, :, max_tumor_slice]
mask_slice = mask_data[:, :, max_tumor_slice]

print(f"   Scan slice - Min: {scan_slice.min()}, Max: {scan_slice.max()}")

# Apply mask and check HU values
mask_binary = (mask_slice > 0).astype(float)
tumor_pixels = scan_slice[mask_binary > 0]

if len(tumor_pixels) > 0:
    print(f"   Tumor HU values:")
    print(f"      Min: {tumor_pixels.min()}")
    print(f"      Max: {tumor_pixels.max()}")
    print(f"      Mean: {tumor_pixels.mean():.1f}")
    print(f"      Median: {np.median(tumor_pixels):.1f}")
else:
    print("   ❌ No tumor pixels after masking!")

# Test different HU windows
print(f"\n🪟 Testing Different HU Windows:")
windows = [
    (-50, 300, "Default"),
    (-100, 400, "Wider"),
    (-200, 500, "Very Wide"),
    (0, 1000, "Original paper"),
]

for min_hu, max_hu, name in windows:
    windowed = np.where(tumor_pixels < min_hu, 0, tumor_pixels)
    windowed = np.where(windowed > max_hu, 0, windowed)
    count = np.count_nonzero(windowed)
    percentage = (count / len(tumor_pixels)) * 100 if len(tumor_pixels) > 0 else 0
    print(f"   {name} ({min_hu} to {max_hu}): {count}/{len(tumor_pixels)} pixels ({percentage:.1f}%)")

# Visualize histogram of tumor HU values
print(f"\n📊 Tumor HU Value Distribution:")
if len(tumor_pixels) > 0:
    bins = [-1000, -500, -200, -100, -50, 0, 50, 100, 200, 300, 500, 1000, 3000]
    hist, _ = np.histogram(tumor_pixels, bins=bins)
    for i in range(len(bins)-1):
        print(f"   {bins[i]:5d} to {bins[i+1]:5d}: {hist[i]:6d} pixels")

print(f"\n{'='*70}")
print("💡 RECOMMENDATIONS:")
print(f"{'='*70}")

if len(tumor_pixels) == 0:
    print("❌ Mask is not aligning with scan!")
    print("   Solution: Check if mask needs to be reoriented differently")
elif tumor_pixels.min() < -50 and tumor_pixels.max() < 300:
    print("⚠️  Tumor HU values are mostly BELOW -50!")
    print(f"   Current range: {tumor_pixels.min():.0f} to {tumor_pixels.max():.0f}")
    print("   Solution: Use wider HU window, e.g., (-200, 300)")
elif tumor_pixels.min() > 300:
    print("⚠️  Tumor HU values are ABOVE 300!")
    print("   Solution: Use higher HU window, e.g., (-50, 500)")
else:
    print("✅ HU values look normal")
    print("   Check if mask binarization is correct")

print(f"{'='*70}\n")


🔍 DEBUGGING PATIENT: 965445P

📂 Files:
   Scan: ./convert_nifti_format/965445P/image_re.nii.gz
   Mask: ./convert_nifti_format/965445P/mask_GTVp_re.nii.gz

📐 Shapes:
   Scan: (512, 512, 186)
   Mask: (512, 512, 186)
   ✅ Shapes match

🎭 Mask Analysis:
   Unique values: [  0. 255.]
   Min: 0.0
   Max: 255.0
   Total non-zero pixels: 396
   Slices with tumor: 3/186
   Slice range: 84 to 86
   Largest tumor on slice: 84 (148 pixels)

🔬 Scan Analysis (slice 84):
   Scan slice - Min: -1024.0, Max: 2261.0
   Tumor HU values:
      Min: -1009.0
      Max: -991.0
      Mean: -999.8
      Median: -1000.0

🪟 Testing Different HU Windows:
   Default (-50 to 300): 0/148 pixels (0.0%)
   Wider (-100 to 400): 0/148 pixels (0.0%)
   Very Wide (-200 to 500): 0/148 pixels (0.0%)
   Original paper (0 to 1000): 0/148 pixels (0.0%)

📊 Tumor HU Value Distribution:
   -1000 to  -500:     88 pixels
    -500 to  -200:      0 pixels
    -200 to  -100:      0 pixels
    -100 to   -50:      0 pixels
     -50 to